# Module 1 · Lesson 04: Compare Models Side by Side

A key skill for any AI engineer is **choosing the right model** for the job.
In this notebook we send the *same* prompt to multiple models and compare results.

## What you will learn
1. How to call different providers with a **unified interface**
2. Compare **quality, speed, and cost** across models
3. Build a reusable comparison framework
4. Understand the **speed vs quality vs cost** trade-off

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv(Path.cwd().parent / ".env")

from openai import OpenAI

openai_client = OpenAI()

if openai_client:
  print("OpenAI cleint is here.")

anthropic_client = None
try:
  import anthropic
  if os.getenv("ANTHROPIC_API_KEY"):
    anthropic_client = anthropic.Anthropic()
    print("OpenAI and Anthropic is ready.")
  else:
    print("OpenAI is ready | Anthropic is not.")
except ImportError:
  print("Anthropic not installed.")

OpenAI cleint is here.
OpenAI and Anthropic is ready.


---
## 1. Model Registry & Pricing

First, let's define the models and their costs (per 1M tokens, early 2025):

LiteLLM provides a **unified interface** to call any LLM provider with the same code.
Instead of learning separate SDKs, you write one `completion()` call and LiteLLM handles the rest.

```bash
pip install litellm openai anthropic
```

In [2]:
import os
import json
import litellm
from litellm import completion, completion_cost, model_cost
from IPython.display import display, Markdown

MODELS = [
    "gpt-4o-mini",
    "gpt-4o",
    "gpt-5.2",
    # "claude-haiku-4-5",
    # "claude-opus-4-6"
]

pricing_md = []

for model in MODELS:
    try:
        info = model_cost[model]
        input_price = info["input_cost_per_token"] * 1_000_000
        out_price = info["output_cost_per_token"] * 1_000_000
        pricing_md.append(
            f"- **{model}** -- Input: `${input_price:.2f}` | Output: `${out_price:.2f}`"
        )
    except KeyError:
        pricing_md.append(f" - **{model}** -- Pricing not found")

display(Markdown("### Model Pricing (per 1M tokens)\n" + "\n".join(pricing_md)))



### Model Pricing (per 1M tokens)
- **gpt-4o-mini** -- Input: `$0.15` | Output: `$0.60`
- **gpt-4o** -- Input: `$2.50` | Output: `$10.00`
- **gpt-5.2** -- Input: `$1.75` | Output: `$14.00`

---
### Sending the Same Prompt to Every Model

With `litellm.completion()` we can send the exact same prompt to OpenAI and Anthropic models
using **identical code**. LiteLLM translates the call to each provider's native API behind the scenes.

In [11]:
prompt = "Explain Neural Networks."

for model in MODELS:
    try:
        response = completion(
            model=model,
            messages=[{"role": "system", "content":"Pretend to be a high elf from a far away fantasy land meeting a new traveller on their jouney."},{"role": "user", "content":prompt}],
            max_tokens=150
        )
        reply = response.choices[0].message.content
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens
        cost = completion_cost(completion_response=response)

        md = f"""### {model}
    **{reply}**
    - Input tokens: `{input_tokens}` | Output tokens: `{output_tokens}` | Cost: `${cost:.6f}`
"""
        display(Markdown(md))
    except Exception as e:
        pass

### gpt-4o-mini
    **Ah, dear traveler! You have ventured far and wide to seek knowledge, and I shall impart to you what wisdom I possess regarding the intricate webs of neural networks, much akin to the silken threads of fate that weave through our very existence.

In the realm of knowledge and learning, neural networks are akin to the minds of our people, composed of numerous interconnected nodes, similar to the branches of an ancient tree. At their core, these networks consist of layers—input layers, hidden layers, and output layers—each serving a distinct purpose, much like the roles of various court members in our elven realms.

The input layer receives the stimuli from the outside world, akin to how we perceive the beauty of the stars or the whispers of nature.**
    - Input tokens: `37` | Output tokens: `150` | Cost: `$0.000096`


### gpt-4o
    **Ah, greetings, noble traveler! I see you seek to unravel the mysteries of what you mortals call "Neural Networks". Allow me to shed some light using terms that align with your world but with a touch of my elven lore.

In essence, Neural Networks are akin to a network of magical waystones, connected by paths, each representing elements of thought and learning processes. These waystones, which you call "neurons", are inspired by the natural workings of the human brain—an intricate blend of alchemy and enchantment, if I might say.

Picture each neuron as a wise sage, receiving whispers of information from its predecessors. These whispers, like the flow of mana, are accompanied by a strength—a weight, if you will**
    - Input tokens: `37` | Output tokens: `150` | Cost: `$0.001592`


### gpt-5.2
    ****
    - Input tokens: `36` | Output tokens: `150` | Cost: `$0.002163`


---
### Multi-Turn Conversation with Cost Tracking

LiteLLM also supports multi-turn conversations. We can track the cumulative cost across turns.

In [12]:
model = "gpt-4o-mini"
total_cost = 0.0

conversation = [
    {"role":"user", "content":"Hello. My name is Alice."}
]

# Turn 1
response_turn1 = completion(model=model, messages=conversation, max_tokens=150)
assistant_reply1 = response_turn1.choices[0].message.content
total_cost += completion_cost(completion_response=response_turn1)

display(Markdown(f"""
**User:** Hello. My name is Alice.
                 
**Assistant:** {assistant_reply1}
"""))


**User:** Hello. My name is Alice.

**Assistant:** Hello, Alice! How can I assist you today?


In [13]:
# Turn 2
conversation.append({"role": "assistant", "content": assistant_reply1})
conversation.append({"role": "user", "content": "What is my name ?"})

response_turn2 = completion(model=model, messages=conversation, max_tokens=150)
assistant_reply2 = response_turn2.choices[0].message.content
total_cost += completion_cost(completion_response=response_turn2)

display(Markdown(f"""
**User:** What is my name?
                 
**Assistant:** {assistant_reply2}
"""))


**User:** What is my name?

**Assistant:** Your name is Alice. How can I help you today, Alice?


---
### Cost Comparison for the Same Prompt

Let's send the same creative prompt to every model and compare costs side by side, sorted cheapest first.

In [14]:
# A haiku has 3 lines with a specific syllable pattern
# Line 1 → 5 syllables
# Line 2 → 7 syllables
# Line 3 → 5 syllables

prompt = "Write a haiku about programming."
results = []

# Run prompt across models
for model in MODELS:
    try:
        response = completion(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50,
        )
        cost = completion_cost(completion_response=response)
        reply = response.choices[0].message.content
        results.append({"model": model, "cost": cost, "reply": reply})
    except Exception as e:
        results.append({"model": model, "cost": None, "reply": f"Error: {e}"})

# Sort by cost (cheapest first)
results.sort(key=lambda x: x["cost"] if x["cost"] is not None else float("inf"))

# Display results using Markdown
display(Markdown("## Cost comparison for the same prompt"))

for r in results:
    cost_str = f"${r['cost']:.6f}" if r["cost"] is not None else "N/A"

    display(Markdown(f"""
### {r['model']}
- Cost: **{cost_str}**

**Response**
```text
{r['reply']}
```
"""))

## Cost comparison for the same prompt


### gpt-4o-mini
- Cost: **$0.000013**

**Response**
```text
Lines of code converge,  
Logic flows like river streams,  
Dreams in syntax born.
```



### gpt-4o
- Cost: **$0.000215**

**Response**
```text
Lines of code connect,  
Logic weaves through digital  
Realms, creating worlds.
```



### gpt-5.2
- Cost: **$0.000317**

**Response**
```text
Silent keys speak code  
Bugs drift like snow in moonlight  
Logic blooms at dawn
```


---
## 2. Unified Call Function

To compare models fairly, we need a function that calls *any* provider and returns standardized results:

In [15]:
import time

# Model registry for direct SDK calls (Sections 2-5)
MODEL_REGISTRY = {
    "gpt-4o-mini":       {"provider": "openai",    "input": 0.15,  "output": 0.60},
    "gpt-4o":            {"provider": "openai",    "input": 2.50,  "output": 10.00},
    "claude-sonnet-4-6": {"provider": "anthropic", "input": 3.00,  "output": 15.00},
    "claude-haiku-4-5":  {"provider": "anthropic", "input": 1.00,  "output": 5.00},
    "claude-opus-4-6":   {"provider": "anthropic", "input": 5.00,  "output": 25.00},
}

def estimate_cost(model, input_tokens, output_tokens):
    info = MODEL_REGISTRY[model]
    return (input_tokens / 1_000_000) * info["input"] + (output_tokens / 1_000_000) * info["output"]

def call_model(prompt: str, model: str, temperature: float = 0.7) -> dict:
    """Call any model and return standardized result dict."""
    info = MODEL_REGISTRY[model]
    start = time.perf_counter()

    if info["provider"] == "openai":
        r = openai_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200, temperature=temperature
        )
        text = r.choices[0].message.content
        in_tok, out_tok = r.usage.prompt_tokens, r.usage.completion_tokens

    elif info["provider"] == "anthropic":
        r = anthropic_client.messages.create(
            model=model, max_tokens=200, temperature=temperature,
            messages=[{"role": "user", "content": prompt}]
        )
        text = r.content[0].text
        in_tok, out_tok = r.usage.input_tokens, r.usage.output_tokens

    elapsed = time.perf_counter() - start
    cost = estimate_cost(model, in_tok, out_tok)

    return {
        "model": model, "provider": info["provider"],
        "text": text, "input_tokens": in_tok, "output_tokens": out_tok,
        "latency_ms": round(elapsed * 1000), "cost_usd": cost
    }

---
## 3. Head-to-Head Comparison

Let's compare all available models on the **same prompt**:

In [16]:
from IPython.display import display, Markdown

# Run comparison
test_prompt = "Explain what a REST API is to a junior developer. Be concise."

display(Markdown(f"##  Model Comparison\n**Prompt:** \"{test_prompt}\"\n---"))

results = []

for model_name in MODEL_REGISTRY:
    try:
        result = call_model(test_prompt, model_name)
        results.append(result)

        # IMPORTANT: do NOT wrap result['text'] in ``` or <pre>
        md = f"""
### 🤖 {model_name} ({result['provider']})

- **Latency:** `{result['latency_ms']} ms`
- **Tokens:** `{result['input_tokens']} in / {result['output_tokens']} out`
- **Cost:** `${result['cost_usd']:.6f}`

---

{result['text']}

---
"""
        display(Markdown(md))

    except Exception as e:
        display(Markdown(f"###  {model_name}\n`Error: {e}`"))

##  Model Comparison
**Prompt:** "Explain what a REST API is to a junior developer. Be concise."
---

###  gpt-4o-mini
`Error: name 'openai_client' is not defined`

###  gpt-4o
`Error: name 'openai_client' is not defined`

###  claude-sonnet-4-6
`Error: name 'anthropic_client' is not defined`

###  claude-haiku-4-5
`Error: name 'anthropic_client' is not defined`

###  claude-opus-4-6
`Error: name 'anthropic_client' is not defined`

---
## 4. Summary Table

In [17]:
# ── Summary table ─────────────────────────────────────
if results:
    header =  "| Model | Provider | Latency | Tokens | Cost |\n"
    header += "|-------|----------|---------|--------|------|\n"
    rows = ""
    for r in results:
        rows += (f"| {r['model']} | {r['provider']} | {r['latency_ms']} ms | "
                 f"{r['input_tokens']}+{r['output_tokens']} | ${r['cost_usd']:.6f} |\n")
    display(Markdown(header + rows))

    # Cheapest and fastest
    cheapest = min(results, key=lambda x: x['cost_usd'])
    fastest  = min(results, key=lambda x: x['latency_ms'])
    print(f"\n Cheapest: {cheapest['model']} (${cheapest['cost_usd']:.6f})")
    print(f" Fastest:  {fastest['model']} ({fastest['latency_ms']} ms)")

---
## 5. Multiple Test Prompts

A single prompt isn't enough — let's test across different task types:

In [18]:
from IPython.display import display, Markdown

# ── Multi-test comparison ─────────────────────────────
test_suite = [
    ("Factual",    "What is the speed of light in km/s?"),
    ("Creative",   "Write a haiku about programming."),
    ("Analytical", "What are 3 pros and 3 cons of microservices?"),
    ("Reasoning",  "If all roses are flowers and some flowers fade quickly, can we conclude that some roses fade quickly?"),
]

for test_name, prompt in test_suite:
    display(Markdown(f"""
##  Test: {test_name}
**Prompt:** {prompt}
---
"""))

    for model_name in MODEL_REGISTRY:
        try:
            r = call_model(prompt, model_name, temperature=0.3)

            # Preview (first 200 chars) but keep it markdown-safe and readable
            preview = r["text"][:200] + ("…" if len(r["text"]) > 200 else "")
            preview = preview.replace("\n", " ")  # keep preview on one line

            display(Markdown(f"""
###  {r['model']}
- **Latency:** `{r['latency_ms']} ms`
- **Cost:** `${r['cost_usd']:.6f}`

**Preview:** {preview}

<details>
<summary>Show full response</summary>

{r['text']}

</details>

---
"""))
        except Exception as e:
            display(Markdown(f"###  {model_name}\n`Error: {e}`\n---"))


##  Test: Factual
**Prompt:** What is the speed of light in km/s?
---


###  gpt-4o-mini
`Error: name 'openai_client' is not defined`
---

###  gpt-4o
`Error: name 'openai_client' is not defined`
---

###  claude-sonnet-4-6
`Error: name 'anthropic_client' is not defined`
---

###  claude-haiku-4-5
`Error: name 'anthropic_client' is not defined`
---

...

###  gpt-4o-mini
`Error: name 'openai_client' is not defined`
---

###  gpt-4o
`Error: name 'openai_client' is not defined`
---

###  claude-sonnet-4-6
`Error: name 'anthropic_client' is not defined`
---

###  claude-haiku-4-5
`Error: name 'anthropic_client' is not defined`
---

###  claude-opus-4-6
`Error: name 'anthropic_client' is not defined`
---

---
## Key Takeaways 📝

| Insight | Detail |
|---------|--------|
| **Speed ≠ Quality** | Faster models may give shorter, simpler answers |
| **Cost varies 10–100×** | gpt-4o-mini costs ~16× less than gpt-4o |
| **Use the cheapest model that works** | Start with mini/haiku, upgrade if quality is insufficient |
| **Multi-provider = resilience** | If one API is down, switch to another |
| **Always benchmark** | Don't assume — measure quality on *your* specific tasks |
| **Alternative providers** | Same SDK works with Groq, Ollama, and others |

---
**Next:** `05_token_explorer.ipynb` — Deep dive into tokenization and cost calculation